# ClinVar Single-Variant Embedding Analysis

Compute and analyze embeddings for ClinVar variants across all genomic foundation models.

**Pipeline:**
1. Load ClinVar variants from bundled `clinvar_variants.tsv`
2. Compute embeddings (WT + MUT + delta) per model → `embeddings/clinvar/{model_key}.db`
3. Compute single-variant geometry metrics (effect size, direction, magnitude)
4. Analyze pathogenicity signal across models

**CLI usage (on cluster):**
```bash
conda activate nt && python -m notebooks.processing.process_clinvar --phase embed --env-profile nt
conda activate borzoi && python -m notebooks.processing.process_clinvar --phase embed --env-profile borzoi
# ... repeat for each env
python -m notebooks.processing.process_clinvar --phase metrics
```

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "genebeddings").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from genebeddings import VariantEmbeddingDB
from notebooks.paper_data_config import embeddings_dir
from notebooks.processing.process_clinvar import (
    load_clinvar_df,
    CLINVAR_SOURCE_NAME,
    ID_COL,
)

In [ ]:
# Load ClinVar variants
df = load_clinvar_df()
print(f"Loaded {len(df)} ClinVar variants")
print(f"Columns: {list(df.columns)}")
if "label" in df.columns:
    print(f"\nLabel distribution:\n{df['label'].value_counts()}")
df.head()

## 2. Discover available embedding DBs

In [ ]:
# Find all ClinVar embedding DBs
clinvar_dir = embeddings_dir() / CLINVAR_SOURCE_NAME
print(f"ClinVar embedding dir: {clinvar_dir}")
print(f"Exists: {clinvar_dir.exists()}")

if clinvar_dir.exists():
    db_files = sorted(clinvar_dir.glob("*.db"))
    print(f"\nAvailable models ({len(db_files)}):")
    for db_path in db_files:
        db = VariantEmbeddingDB(str(db_path))
        n = db.count()
        db.close()
        print(f"  {db_path.stem}: {n} embeddings")
else:
    print("\nNo ClinVar embeddings yet. Run the embedding pipeline first:")
    print("  python -m notebooks.processing.process_clinvar --phase embed --env-profile <env>")

## 3. Compute metrics (or load from existing DBs)

Run this cell to compute single-variant geometry metrics from existing embedding DBs.
If embeddings don't exist yet, provide a model to compute them on-the-fly.

In [ ]:
from genebeddings.genebeddings import add_single_variant_metrics

# Collect metrics across all available models
all_metrics = {}

if clinvar_dir.exists():
    for db_path in sorted(clinvar_dir.glob("*.db")):
        model_key = db_path.stem
        if model_key.endswith("_annotated"):
            continue
        print(f"\nLoading metrics from {model_key}...")
        db = VariantEmbeddingDB(str(db_path))
        try:
            annotated = add_single_variant_metrics(
                df,
                db,
                model=None,  # Load from DB, no model needed
                id_col=ID_COL,
                show_progress=True,
                pool="mean",
            )
            all_metrics[model_key] = annotated
            n_valid = annotated["v_l2"].notna().sum()
            print(f"  {n_valid}/{len(annotated)} variants with metrics")
        finally:
            db.close()

print(f"\n{len(all_metrics)} models with metrics")

## 4. Effect size by pathogenicity class

Compare variant effect sizes (L2 norm of delta embedding) between pathogenic and benign variants across models.

In [ ]:
if all_metrics and "label" in df.columns:
    from scipy.stats import mannwhitneyu

    rows = []
    for model_key, annotated in all_metrics.items():
        for metric in ["v_l2", "v_diff"]:
            if metric not in annotated.columns:
                continue
            pathogenic = annotated.loc[annotated["label"].str.lower() == "pathogenic", metric].dropna()
            benign = annotated.loc[annotated["label"].str.lower() == "benign", metric].dropna()
            if len(pathogenic) < 5 or len(benign) < 5:
                continue
            stat, p = mannwhitneyu(pathogenic, benign, alternative="two-sided")
            rows.append({
                "model": model_key,
                "metric": metric,
                "pathogenic_median": pathogenic.median(),
                "benign_median": benign.median(),
                "p_value": p,
                "n_pathogenic": len(pathogenic),
                "n_benign": len(benign),
            })

    if rows:
        df_results = pd.DataFrame(rows).sort_values("p_value")
        print(df_results.to_string(index=False))
    else:
        print("No results — need 'label' column with 'pathogenic'/'benign' values")
else:
    print("Skipping: no metrics or no 'label' column in data")

## 5. Cross-model comparison

Distribution of effect sizes per model, split by label.

In [ ]:
if all_metrics and "label" in df.columns:
    # Build long-form dataframe for plotting
    plot_rows = []
    for model_key, annotated in all_metrics.items():
        if "v_l2" not in annotated.columns:
            continue
        for _, row in annotated.iterrows():
            if pd.notna(row.get("v_l2")) and pd.notna(row.get("label")):
                plot_rows.append({
                    "model": model_key,
                    "v_l2": row["v_l2"],
                    "label": row["label"],
                })
    if plot_rows:
        df_plot = pd.DataFrame(plot_rows)
        n_models = df_plot["model"].nunique()
        fig, ax = plt.subplots(figsize=(max(8, n_models * 1.2), 5))
        sns.boxplot(data=df_plot, x="model", y="v_l2", hue="label", ax=ax)
        ax.set_ylabel("Effect size (L2 norm of delta)")
        ax.set_xlabel("Model")
        ax.set_title("ClinVar variant effect sizes by model and pathogenicity")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("Skipping plot: no metrics or no label column")